## 1. Load Data

In [1]:
import pandas as pd

# Load the full Bank Marketing dataset
df = pd.read_csv("data/bank-additional-full.csv", sep=";")

# Preview the first five rows
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Basic Data Inspection

In [2]:
# Check dataset dimensions
print("Shape:", df.shape)

# Display all column names
print("\nColumns:")
print(df.columns.tolist())

Shape: (41188, 21)

Columns:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


In [3]:
# Inspect data types and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [4]:
# Check target class counts
print("Target counts:")
print(df["y"].value_counts())

# Check target class proportions
print("\nTarget proportions:")
print(df["y"].value_counts(normalize=True))

Target counts:
y
no     36548
yes     4640
Name: count, dtype: int64

Target proportions:
y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64


## 3. Data Quality Analysis

In [5]:
# Count standard missing values in each column
missing_counts = df.isnull().sum()

missing_counts

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

In [6]:
# Count "unknown" categorical values in each column
unknown_counts = (df == "unknown").sum()

# Show only columns containing unknown values
unknown_counts[unknown_counts > 0]

job           330
marital        80
education    1731
default      8597
housing       990
loan          990
dtype: int64

In [7]:
# Calculate percentage of unknown values
unknown_percent = unknown_counts[unknown_counts > 0] / len(df) * 100

unknown_percent.round(2)

job           0.80
marital       0.19
education     4.20
default      20.87
housing       2.40
loan          2.40
dtype: float64

In [8]:
# Summarise class distribution
class_distribution = pd.DataFrame({
    "count": df["y"].value_counts(),
    "percentage": df["y"].value_counts(normalize=True) * 100
})

class_distribution.round(2)

,count,percentage
y,,
no,36548,88.73
yes,4640,11.27


In [9]:
# Count exact duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 12


In [10]:
# Distribution of previous campaign contacts
df["previous"].value_counts().sort_index()

previous
0    35563
1     4561
2      754
3      216
4       70
5       18
6        5
7        1
Name: count, dtype: int64

In [11]:
# Distribution of previous campaign outcomes
df["poutcome"].value_counts()

poutcome
nonexistent    35563
failure         4252
success         1373
Name: count, dtype: int64

In [12]:
# Compare previous contact counts with previous campaign outcomes
pd.crosstab(df["previous"], df["poutcome"])

poutcome,failure,nonexistent,success
previous,,,
0,0,35563,0
1,3696,0,865
2,434,0,320
3,88,0,128
4,30,0,40
5,3,0,15
6,1,0,4
7,0,0,1


In [13]:
# Inspect pdays values
df["pdays"].value_counts().sort_index()

pdays
0         15
1         26
2         61
3        439
4        118
5         46
6        412
7         60
8         18
9         64
10        52
11        28
12        58
13        36
14        20
15        24
16        11
17         8
18         7
19         3
20         1
21         2
22         3
25         1
26         1
27         1
999    39673
Name: count, dtype: int64

In [14]:
# Count records using the special pdays value 999
pdays_999_count = (df["pdays"] == 999).sum()

print("Number of pdays = 999:", pdays_999_count)
print("Percentage:", round(pdays_999_count / len(df) * 100, 2), "%")

Number of pdays = 999: 39673
Percentage: 96.32 %


In [15]:
# Previous contacts exist, but pdays is still 999
previous_with_999 = (
    (df["previous"] > 0) &
    (df["pdays"] == 999)
).sum()

print("Previous > 0 and pdays = 999:", previous_with_999)

Previous > 0 and pdays = 999: 4110


In [16]:
# No previous contacts, but pdays has a recorded value
no_previous_with_pdays = (
    (df["previous"] == 0) &
    (df["pdays"] != 999)
).sum()

print("Previous = 0 and pdays != 999:", no_previous_with_pdays)

Previous = 0 and pdays != 999: 0


In [17]:
# Compare previous campaign outcome with current subscription outcome
poutcome_target = pd.crosstab(
    df["poutcome"],
    df["y"],
    normalize="index"
)

poutcome_target

y,no,yes
poutcome,,
failure,0.857714,0.142286
nonexistent,0.911678,0.088322
success,0.348871,0.651129


## 4. Remove Duplicate Rows

In [18]:
# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Confirm the updated dataset size
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (41176, 21)


In [19]:
# Confirm no duplicate rows remain
print("Remaining duplicates:", df.duplicated().sum())

Remaining duplicates: 0


## 5 Define RQ1 Feature Set

RQ1 uses a broader feature set to compare the predictive performance of different machine learning models.

The `duration` feature is excluded because it is only known after the call has ended. The original `pdays` variable is replaced by the engineered `pdays_recorded` and `pdays_clean` features.

In [20]:
# Mark whether pdays contains an actual day value
df["pdays_recorded"] = (df["pdays"] != 999).astype(int)

# Replace the sentinel value 999 with 0
df["pdays_clean"] = df["pdays"].where(df["pdays"] != 999, 0)

In [21]:
# Full feature set for RQ1 model comparison
rq1_features = [
    # Demographic and financial
    "age",
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",

    # Current campaign
    "contact",
    "month",
    "day_of_week",
    "campaign",

    # Previous campaign history
    "previous",
    "poutcome",
    "pdays_recorded",
    "pdays_clean",

    # Socio-economic context
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed"
]

# Create RQ1 feature matrix
X_rq1 = df[rq1_features].copy()

print("RQ1 feature count:", len(rq1_features))
print("RQ1 shape:", X_rq1.shape)

RQ1 feature count: 20
RQ1 shape: (41176, 20)


## 6. Define RQ2 Feature Groups

To address RQ2, two feature sets are defined:

- **Set A:** Customer demographic and financial information only.
- **Set B:** Set A plus previous campaign history.

The comparison between these two sets will be used to measure the additional predictive value of previous campaign history.

In [22]:
# Demographic and financial features
base_features = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan"
]

print("Base features:")
print(base_features)

Base features:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']


In [23]:
# Previous campaign history features
history_features = [
    "previous",
    "poutcome",
    "pdays_recorded",
    "pdays_clean"
]

print("History features:")
print(history_features)

History features:
['previous', 'poutcome', 'pdays_recorded', 'pdays_clean']


In [24]:
# Set A: demographic + financial information
features_without_history = base_features

# Set B: base features + previous campaign history
features_with_history = base_features + history_features

print("Set A - Without history:")
print(features_without_history)

print("\nSet B - With history:")
print(features_with_history)

Set A - Without history:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']

Set B - With history:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'previous', 'poutcome', 'pdays_recorded', 'pdays_clean']


In [25]:
# Create the two RQ2 feature matrices
X_without_history = df[features_without_history].copy()
X_with_history = df[features_with_history].copy()

# Encode target: no = 0, yes = 1
y = df["y"].map({"no": 0, "yes": 1})

print("Set A shape:", X_without_history.shape)
print("Set B shape:", X_with_history.shape)
print("Target shape:", y.shape)

Set A shape: (41176, 7)
Set B shape: (41176, 11)
Target shape: (41176,)


In [26]:
# Check target encoding
y.value_counts()

y
0    36537
1     4639
Name: count, dtype: int64

In [27]:
# Preview both feature sets
print("Set A:")
display(X_without_history.head())

print("Set B:")
display(X_with_history.head())

Set A:


,age,job,marital,education,default,housing,loan
0,56,housemaid,married,basic.4y,no,no,no
1,57,services,married,high.school,unknown,no,no
2,37,services,married,high.school,no,yes,no
3,40,admin.,married,basic.6y,no,no,no
4,56,services,married,high.school,no,no,yes


Set B:


,age,job,marital,education,default,housing,loan,previous,poutcome,pdays_recorded,pdays_clean
0,56,housemaid,married,basic.4y,no,no,no,0,nonexistent,0,0
1,57,services,married,high.school,unknown,no,no,0,nonexistent,0,0
2,37,services,married,high.school,no,yes,no,0,nonexistent,0,0
3,40,admin.,married,basic.6y,no,no,no,0,nonexistent,0,0
4,56,services,married,high.school,no,no,yes,0,nonexistent,0,0


## 7. Train/Test Split

The data is split into 80% training and 20% test sets using stratified sampling to preserve the class distribution of the target variable.

The same train/test observations are used for RQ1 and both RQ2 feature sets to ensure consistent and fair model comparisons.

In [28]:
from sklearn.model_selection import train_test_split

# Split row indices once so both feature sets use identical samples
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(train_idx))
print("Test samples:", len(test_idx))

Training samples: 32940
Test samples: 8236


In [29]:
# Set A: without previous campaign history
X_train_without = X_without_history.loc[train_idx].copy()
X_test_without = X_without_history.loc[test_idx].copy()

# Set B: with previous campaign history
X_train_with = X_with_history.loc[train_idx].copy()
X_test_with = X_with_history.loc[test_idx].copy()

# RQ1: full feature set
X_train_rq1 = X_rq1.loc[train_idx].copy()
X_test_rq1 = X_rq1.loc[test_idx].copy()

# Target
y_train = y.loc[train_idx].copy()
y_test = y.loc[test_idx].copy()

In [30]:
# Check train/test dimensions
print("RQ1 train:", X_train_rq1.shape)
print("RQ1 test:", X_test_rq1.shape)

print("\nSet A train:", X_train_without.shape)
print("Set A test:", X_test_without.shape)

print("\nSet B train:", X_train_with.shape)
print("Set B test:", X_test_with.shape)

print("\ny train:", y_train.shape)
print("y test:", y_test.shape)

RQ1 train: (32940, 20)
RQ1 test: (8236, 20)

Set A train: (32940, 7)
Set A test: (8236, 7)

Set B train: (32940, 11)
Set B test: (8236, 11)

y train: (32940,)
y test: (8236,)


In [31]:
# Check class proportions after splitting
print("Training target proportions:")
print(y_train.value_counts(normalize=True))

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True))

Training target proportions:
y
0    0.887341
1    0.112659
Name: proportion, dtype: float64

Test target proportions:
y
0    0.887324
1    0.112676
Name: proportion, dtype: float64


In [32]:
# Confirm all feature sets use the same rows
print(
    "RQ1 and Set A same training rows:",
    X_train_rq1.index.equals(X_train_without.index)
)

print(
    "RQ1 and Set B same training rows:",
    X_train_rq1.index.equals(X_train_with.index)
)

print(
    "RQ1 and Set A same test rows:",
    X_test_rq1.index.equals(X_test_without.index)
)

print(
    "RQ1 and Set B same test rows:",
    X_test_rq1.index.equals(X_test_with.index)
)

RQ1 and Set A same training rows: True
RQ1 and Set B same training rows: True
RQ1 and Set A same test rows: True
RQ1 and Set B same test rows: True


## 8. Preprocessing

Categorical features are one-hot encoded, while numerical features are standardised.

The `"unknown"` values are retained as valid categorical levels because removing these records would discard a substantial portion of the dataset.

All preprocessing transformations are fitted only on the training data and then applied to the test data to avoid data leakage.

In [33]:
# Set A categorical features
categorical_without = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan"
]

# Set A numerical features
numerical_without = [
    "age"
]

In [34]:
# Set B categorical features
categorical_with = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "poutcome"
]

# Set B numerical features
numerical_with = [
    "age",
    "previous",
    "pdays_clean"
]

# Binary indicator does not require scaling
binary_with = [
    "pdays_recorded"
]

In [35]:
# RQ1 categorical features
categorical_rq1 = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome"
]

# RQ1 numerical features
numerical_rq1 = [
    "age",
    "campaign",
    "previous",
    "pdays_clean",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed"
]

# RQ1 binary feature
binary_rq1 = [
    "pdays_recorded"
]

print(
    "RQ1 total raw features:",
    len(categorical_rq1) + len(numerical_rq1) + len(binary_rq1)
)

RQ1 total raw features: 20


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Preprocessor for Set A
preprocessor_without = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_without
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_without
        )
    ]
)

In [37]:
# Preprocessor for Set B
preprocessor_with = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_with
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_with
        ),
        (
            "binary",
            "passthrough",
            binary_with
        )
    ]
)

In [38]:
# Preprocessor for RQ1
preprocessor_rq1 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_rq1
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_rq1
        ),
        (
            "binary",
            "passthrough",
            binary_rq1
        )
    ]
)

In [39]:
# Fit preprocessing on Set A training data
X_train_without_processed = preprocessor_without.fit_transform(
    X_train_without
)

# Apply the same preprocessing to Set A test data
X_test_without_processed = preprocessor_without.transform(
    X_test_without
)

In [40]:
# Fit preprocessing on Set B training data
X_train_with_processed = preprocessor_with.fit_transform(
    X_train_with
)

# Apply the same preprocessing to Set B test data
X_test_with_processed = preprocessor_with.transform(
    X_test_with
)

In [41]:
# Fit preprocessing on RQ1 training data
X_train_rq1_processed = preprocessor_rq1.fit_transform(
    X_train_rq1
)

# Apply the same preprocessing to RQ1 test data
X_test_rq1_processed = preprocessor_rq1.transform(
    X_test_rq1
)

In [42]:
# Check processed dimensions
print("RQ1 train:", X_train_rq1_processed.shape)
print("RQ1 test:", X_test_rq1_processed.shape)

print("\nSet A train:", X_train_without_processed.shape)
print("Set A test:", X_test_without_processed.shape)

print("\nSet B train:", X_train_with_processed.shape)
print("Set B test:", X_test_with_processed.shape)

RQ1 train: (32940, 63)
RQ1 test: (8236, 63)

Set A train: (32940, 34)
Set A test: (8236, 34)

Set B train: (32940, 40)
Set B test: (8236, 40)


In [43]:
# Get processed feature names
feature_names_rq1 = preprocessor_rq1.get_feature_names_out()
feature_names_without = preprocessor_without.get_feature_names_out()
feature_names_with = preprocessor_with.get_feature_names_out()

print("RQ1 feature count:", len(feature_names_rq1))
print(feature_names_rq1)

print("\nSet A feature count:", len(feature_names_without))
print(feature_names_without)

print("\nSet B feature count:", len(feature_names_with))
print(feature_names_with)

RQ1 feature count: 63
['categorical__job_admin.' 'categorical__job_blue-collar'
 'categorical__job_entrepreneur' 'categorical__job_housemaid'
 'categorical__job_management' 'categorical__job_retired'
 'categorical__job_self-employed' 'categorical__job_services'
 'categorical__job_student' 'categorical__job_technician'
 'categorical__job_unemployed' 'categorical__job_unknown'
 'categorical__marital_divorced' 'categorical__marital_married'
 'categorical__marital_single' 'categorical__marital_unknown'
 'categorical__education_basic.4y' 'categorical__education_basic.6y'
 'categorical__education_basic.9y' 'categorical__education_high.school'
 'categorical__education_illiterate'
 'categorical__education_professional.course'
 'categorical__education_university.degree'
 'categorical__education_unknown' 'categorical__default_no'
 'categorical__default_unknown' 'categorical__default_yes'
 'categorical__housing_no' 'categorical__housing_unknown'
 'categorical__housing_yes' 'categorical__loan_no'


In [44]:
# Convert processed RQ1 data to DataFrames
X_train_rq1_processed = pd.DataFrame(
    X_train_rq1_processed,
    columns=feature_names_rq1,
    index=X_train_rq1.index
)

X_test_rq1_processed = pd.DataFrame(
    X_test_rq1_processed,
    columns=feature_names_rq1,
    index=X_test_rq1.index
)

# Convert processed Set A to DataFrames
X_train_without_processed = pd.DataFrame(
    X_train_without_processed,
    columns=feature_names_without,
    index=X_train_without.index
)

X_test_without_processed = pd.DataFrame(
    X_test_without_processed,
    columns=feature_names_without,
    index=X_test_without.index
)

# Convert processed Set B to DataFrames
X_train_with_processed = pd.DataFrame(
    X_train_with_processed,
    columns=feature_names_with,
    index=X_train_with.index
)

X_test_with_processed = pd.DataFrame(
    X_test_with_processed,
    columns=feature_names_with,
    index=X_test_with.index
)

In [45]:
# Final preprocessing checks
print("RQ1 missing values:",
      X_train_rq1_processed.isnull().sum().sum())

print("Set A missing values:",
      X_train_without_processed.isnull().sum().sum())

print("Set B missing values:",
      X_train_with_processed.isnull().sum().sum())

print(
    "RQ1 train rows match target:",
    X_train_rq1_processed.index.equals(y_train.index)
)

print(
    "RQ1 test rows match target:",
    X_test_rq1_processed.index.equals(y_test.index)
)

print(
    "Set A and Set B same training rows:",
    X_train_without_processed.index.equals(
        X_train_with_processed.index
    )
)

print(
    "Set A and Set B same test rows:",
    X_test_without_processed.index.equals(
        X_test_with_processed.index
    )
)

RQ1 missing values: 0
Set A missing values: 0
Set B missing values: 0
RQ1 train rows match target: True
RQ1 test rows match target: True
Set A and Set B same training rows: True
Set A and Set B same test rows: True


## 9. RQ2 Experimental Setup

RQ2 evaluates the additional predictive value of previous campaign history by comparing two feature sets:

- **Set A:** Demographic and financial information only.
- **Set B:** Set A plus previous campaign history.

For a fair comparison, both feature sets use the same train/test observations, validation strategy, model configuration, and evaluation metrics.

In [46]:
# Summary of the two RQ2 feature sets
rq2_feature_sets = {
    "Set A - Without History": {
        "raw_features": features_without_history,
        "processed_feature_count": len(feature_names_without)
    },
    "Set B - With History": {
        "raw_features": features_with_history,
        "processed_feature_count": len(feature_names_with)
    }
}

for name, info in rq2_feature_sets.items():
    print(name)
    print("Raw features:", info["raw_features"])
    print("Processed feature count:", info["processed_feature_count"])
    print()

Set A - Without History
Raw features: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
Processed feature count: 34

Set B - With History
Raw features: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'previous', 'poutcome', 'pdays_recorded', 'pdays_clean']
Processed feature count: 40



In [47]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Evaluate a binary classification model
def evaluate_model(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

In [48]:
# Store RQ2 model comparison results
rq2_results = pd.DataFrame(
    columns=[
        "Model",
        "Feature Set",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ]
)

rq2_results

,Model,Feature Set,Accuracy,Precision,Recall,F1,ROC-AUC
